#### Explaining RISC-V Code

In [1]:
import import_ipynb
from P0 import compileString

Generate the RISC-V assembly code of the following P0 program:

In [2]:
compileString("""
var a: integer
var b: [0 .. 9] → integer
procedure q(c: integer)
  var d: integer
  var e: [0 .. 9] → integer
    a := c
    d := a
    e[c] := b[a]
program p
  q(3)
""", target = 'riscv')

	.data
a_:	.space 4
b_:	.space 40
	.text
	.globl q
q:	
	addi sp, sp, -64
	sw ra, 56(sp)
	sw s0, 52(sp)
	addi s0, sp, 60
	lw s5, 0(s0)
	la s11, a_
	sw s5, 0(s11)
	la s4, a_
	lw s9, 0(s4)
	sw s9, -12(s0)
	addi s2, zero, 4
	mul s5, s5, s2
	add s5, s0, s5
	addi s10, zero, 4
	mul s9, s9, s10
	lw s6, b_(s9)
	sw s6, -52(s5)
	lw ra, 56(sp)
	lw s0, 52(sp)
	addi sp, sp, 64
	ret
	.globl main
main:	
	jal ra, p
	addi a0, zero, 0
	addi a7, zero, 93
	scall
	.globl p
p:	
	addi sp, sp, -16
	sw ra, 12(sp)
	sw s0, 8(sp)
	addi s0, sp, 16
	addi s5, zero, 3
	sw s5, -4(sp)
	jal ra, q
	lw ra, 12(sp)
	lw s0, 8(sp)
	addi sp, sp, 16
	ret


In the RISV-V code generator of the P0 compiler (or any compiler for a RISC architecture)
- where and when are global integer and array variables allocated,
- where and when are local integer and array variables allocated?

Give brief answers and relate your answer to the generated code above!

**Global integer and array variables.** Allocated at **compile time**, in the static data segment (`.data`). The code generator emits one label per global with `.space N`, where `N` is the type's byte size. In the output above:

```
.data
a_:  .space 4       # global integer a → 4 bytes
b_:  .space 40      # global array b[0..9] → 10 × 4 = 40 bytes
```

Accesses use the label as a symbolic address: `la s11, a_; sw s5, 0(s11)` implements `a := c`, and array indexing uses the label as a base — `mul s9, s9, 4; lw s6, b_(s9)` computes `s6 := b[a]`. Because globals are static, their addresses are resolved by the assembler/linker, not during execution.

**Local integer and array variables.** Allocated at **run time**, on the stack, by the prologue of their enclosing procedure. The code generator assigns each local a fixed offset relative to the frame pointer `s0` and reserves total frame space with a single `addi sp, sp, -framesize`. In procedure `q`:

- the prologue `addi sp, sp, -64` claims 64 bytes for the whole frame,
- `d: integer` lives at `-12(s0)` (4 bytes) — the store `sw s9, -12(s0)` implements `d := a`,
- `e: [0..9] → integer` has its base at `-52(s0)` (40 bytes, slots `-52(s0)` through `-16(s0)`). Indexing `e[c]` is compiled as `mul s5, c, 4; add s5, s0, s5; sw s6, -52(s5)` — i.e. `(s0 + c×4) + (-52)` addresses `e[c]`,
- the epilogue `addi sp, sp, 64` releases the frame, so locals' storage lifetimes coincide with the call's activation.

Explain the role of the stack pointer (`sp`) and frame pointer (`s0`) in RISV-V code generated by the P0 compiler (or any compiler for a RISC architecture). Relate your answer to the generated code above!

**Stack pointer `sp`.** Points to the top of the currently-active stack (the lowest live address, since the stack grows downward). Two jobs:

1. **Allocate/free the current frame.** Each procedure's prologue subtracts its frame size from `sp` and the epilogue adds it back. In the output: `q` uses `addi sp, sp, -64 … addi sp, sp, 64`; `p` uses `addi sp, sp, -16 … addi sp, sp, 16`.
2. **Pass stack-resident arguments.** Before a call, the caller writes outgoing parameters just below its own `sp`. In `p`, `addi s5, zero, 3; sw s5, -4(sp)` writes the argument `3` into `sp-4`, then `jal ra, q` transfers control. After `q`'s prologue moves `sp` down by 64 and sets `s0 = sp + 60`, the very same memory word is reachable from the callee as `0(s0)` — so `lw s5, 0(s0)` fetches the parameter `c`.

**Frame pointer `s0`.** A stable anchor for the current activation: set once in the prologue (`addi s0, sp, 60`) and untouched by the body, even if a nested call later shifts `sp`. Its job is to give every local, parameter, and saved register a fixed, predictable offset independent of intervening pushes and pops.

Frame layout in `q` (relative to `s0 = sp + 60`):

| offset | content | lived in |
|:---|:---|:---|
| `0(s0)` | parameter `c` | caller (written before `jal`) |
| `-4(s0)` i.e. `56(sp)` | saved `ra` | `sw ra, 56(sp)` |
| `-8(s0)` i.e. `52(sp)` | saved `s0` | `sw s0, 52(sp)` |
| `-12(s0)` | local `d` | `sw s9, -12(s0)` |
| `-52(s0)` … `-16(s0)` | local array `e[0..9]` | `sw s6, -52(s5)` for `e[c]` |

Because `s0` is callee-saved, the epilogue must restore it: `lw s0, 52(sp)` before `ret`. `sp` is by contrast only used for the coarse, frame-granularity moves — it changes whenever a call or return happens, but `s0` remains valid for the entire body of the current procedure.

---

Here are some RISC-V instructions:

| Instruction         | Effect                                                 | Description              | 
|:--------------------|:-------------------------------------------------------|:-------------------------|
| `add rd, rs1, rs2`  | `R[rd], pc := (𝙪R[rs1] + 𝙪R[rs2]) 𝗺𝗼𝗱 2³², pc + 4`     | addition⁽¹⁾              |
| `sub rd, rs1, rs2`  | `R[rd], pc := (𝙪R[rs1] – 𝙪R[rs2]) 𝗺𝗼𝗱 2³², pc + 4`     | subtraction⁽¹⁾           |
| `and rd, rs1, rs2`  | `R[rd], pc := R[rs1] & R[rs2], pc + 4`                 | bitwise and⁽²⁾           |
| `or rd, rs1, rs2`   | `R[rd], pc := R[rs1] \| R[rs2], pc + 4`                | bitwise or⁽³⁾            |
| `xor rd, rs1, rs2`  | `R[rd], pc := R[rs1] ^ R[rs2], pc + 4`                 | bitwise xor⁽⁴⁾           |
| `sll rd, rs1, rs2`  | `R[rd], pc := R[rs1] 𝘀𝗹𝗹 𝙪R[rs2], pc + 4`               | shift left⁽⁵⁾             |
| `sra rd, rs1, rs2`  | `R[rd], pc := 1 if R[rs1] 𝘀𝗿𝗮 𝙪R[rs2]) else 0, pc + 4`  | shift right arithmetic⁽⁶⁾ |
| `srl rd, rs1, rs2`  | `R[rd], pc := 1 if R[rs1] 𝘀𝗿𝗹 𝙪R[rs2]) else 0, pc + 4`  | shift right (logical)⁽⁷⁾  |
| `slt rd, rs1, rs2`  | `R[rd], pc := 1 if 𝙨R[rs1]) < 𝙨R[rs2]) else 0, pc + 4` | signed comparison         |
| `sltu rd, rs1, rs2` | `R[rd], pc := 1 if 𝙪R[rs1] < 𝙪R[rs2] else 0, pc + 4`   | unsigned comparison      |

1) Addition and subtraction interpret registers as unsigned integers. The result is also correct if the registers hold signed integers in two's complement and the result fits in a word.
2) For words `a` and `b`: `(a & b)[i] = 1 𝗶𝗳 a[i] = 1 ∧ b[i] = 1 𝗲𝗹𝘀𝗲 0`, for `0 ≤ i < 32`.
3) For words `a` and `b`: `(a | b)[i] = 1 𝗶𝗳 a[i] = 1 ∨ b[i] = 1 𝗲𝗹𝘀𝗲 0`, for `0 ≤ i < 32`.
4) For words `a` and `b`: `(a ^ b)[i] = 1 𝗶𝗳 a[i] ≠ b[i] 𝗲𝗹𝘀𝗲 0`, for `0 ≤ i < 32`.
5) For word `a`: `(a 𝘀𝗹𝗹 n)[i] = a[i – n]`, for `n ≤ i < 32` and `(a 𝘀𝗹𝗹 n)[i] = 0` for `0 ≤ i < n` and for `0 ≤ n < 32`.
6) For word `a`: `(a 𝘀𝗿𝗮 n)[i] = a[i + n]`, for `0 ≤ i < 32 – n` and `(a 𝘀𝗿𝗮 n)[i] = a[31]` for `32 – n ≤ i < 32` and for `0 ≤ n < 32`.
7) For word `a`: `(a 𝘀𝗿𝗹 n)[i] = a[i + n]`, for `0 ≤ i < 32 – n` and `(a 𝘀𝗿𝗹 n)[i] = 0` for `32 – n ≤ i < 32` and for `0 ≤ n < 32`.